# SuperNNova transient classifier

This notebook trains the same seven-class, group-safe experiment as the ParSNIP baseline with SuperNNova's bidirectional LSTM. The default two-epoch smoke run uses only training-side folds and produces preliminary validation and disposable smoke-test confusion matrices for photometry-only and exact-redshift variants. Set `EXECUTION_MODE` to a full mode only after reviewing the smoke timing.

In [ ]:
# Import the shared classifier workflow, SuperNNova backend, and plotting packages.
from pathlib import Path
import importlib
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Locate this checkout from the repository root, package, or notebook directory.
cwd = Path.cwd().resolve()
search_roots = (cwd, *cwd.parents)
PROJECT_ROOT = next((root for root in search_roots if (root / "warpTemplate" / "classification.py").exists()), None)
if PROJECT_ROOT is None:
    package_root = next((root for root in search_roots if root.name == "warpTemplate" and (root / "classification.py").exists()), None)
    PROJECT_ROOT = package_root.parent if package_root is not None else None
if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not locate warp_templates above {cwd}")
# Resolve SuperNNova from the kernel while hiding the workspace reference copy.
original_sys_path = list(sys.path)
kernel_sys_path = []
for entry in original_sys_path:
    try:
        resolved_entry = Path(entry or Path.cwd()).resolve()
    except (OSError, RuntimeError):
        resolved_entry = None
    if resolved_entry != PROJECT_ROOT:
        kernel_sys_path.append(entry)
for module_name in [name for name in sys.modules if name == "supernnova" or name.startswith("supernnova.")]:
    sys.modules.pop(module_name, None)
try:
    sys.path[:] = kernel_sys_path
    supernnova_package = importlib.import_module("supernnova")
    training_utils = importlib.import_module("supernnova.utils.training_utils")
finally:
    sys.path[:] = original_sys_path
supernnova_path = Path(supernnova_package.__file__).resolve()
if PROJECT_ROOT in supernnova_path.parents or not hasattr(training_utils, "load_HDF5"):
    raise RuntimeError(
        "The active kernel must provide the astronomical SuperNNova package; "
        f"resolved {supernnova_path}"
    )

# Prefer this checkout only for WarpTemplate itself.
sys.path.insert(0, str(PROJECT_ROOT))

from warpTemplate import classification as workflow
from warpTemplate import supernnova_backend as supernova

sns.set_theme(context="notebook", style="ticks")

## Configuration

`smoke` trains two epochs on 64 objects per class in each temporary role. `full_cpu` uses folds 2–9 for training and fold 1 for validation. `full_gpu_repeats` runs five seeds and refuses to start without CUDA. Official fold-0 evaluation always requires the separate `EVALUATE_TEST` switch.

In [ ]:
# Define the experiment identity, execution budget, and immutable-test safeguards.
SAMPLE_ID = "warp_sample_combined_schema6_56ecb71d1f32"
SAMPLE_DIR = PROJECT_ROOT / "training_samples" / SAMPLE_ID
PACKAGE_ROOT = PROJECT_ROOT / "warpTemplate"
SPLIT_ROOT = PACKAGE_ROOT / "classification_splits" / SAMPLE_ID
RUNS_ROOT = PACKAGE_ROOT / "classifier_runs"
SPLIT_STRATEGY = "basis_sn"
EXECUTION_MODE = "smoke"  # Supported: smoke, full_cpu, full_gpu_repeats.
REDSHIFT_MODES = list(supernova.REDSHIFT_MODES)
SEED = 20260723
THREADS = 14
SMOKE_OBJECTS_PER_CLASS = 128
SMOKE_EPOCHS = 20
FULL_EPOCHS = 90
EVALUATE_TEST = False
ALLOW_TEST_OVERWRITE = False
FORCE_REBUILD_DATABASE = False
FORCE_RETRAIN = False
PARTIAL_CUTOFFS = [-7, -2, -1, 0, 1, 2, 30]

if EXECUTION_MODE not in {"smoke", "full_cpu", "full_gpu_repeats"}:
    raise ValueError(f"Unsupported execution mode: {EXECUTION_MODE}")
if EXECUTION_MODE == "full_gpu_repeats":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("full_gpu_repeats requires CUDA; this machine is CPU-only")
    RUN_SEEDS = list(range(20260721, 20260726))
    DEVICE = "cuda"
else:
    RUN_SEEDS = [SEED]
    DEVICE = "auto"
MAX_EPOCHS = SMOKE_EPOCHS if EXECUTION_MODE == "smoke" else FULL_EPOCHS
workflow.set_random_seed(SEED)
print({"mode": EXECUTION_MODE, "device": DEVICE, "seeds": RUN_SEEDS, "epochs": MAX_EPOCHS})

## Frozen groups and sequence database

Both variants share one raw HDF5 database. It stores all 30 possible channels, while each model selects either 28 photometry channels or those channels plus exact simulated redshift and zero error. Fluxes are converted to zeropoint 27.5, measurements within 0.33 days are grouped, and normalization is learned from the active training role only.

In [ ]:
# Load schema-6 truth and reuse the persistent grouped split manifests.
truth = workflow.load_sample_truth(SAMPLE_DIR)
truth["final_label"] = workflow.merge_fitclasses(truth["fitclass"]).to_numpy()
split_manifests = {}
for strategy in ("template_key", "basis_sn"):
    split_manifests[strategy], excluded = workflow.prepare_persistent_split(
        SAMPLE_DIR, SPLIT_ROOT, strategy=strategy, seed=SEED
    )
split_manifest = split_manifests[SPLIT_STRATEGY]
workflow.validate_split_manifest(split_manifest)
role_manifest = supernova.build_execution_role_manifest(
    split_manifest,
    EXECUTION_MODE,
    objects_per_class=SMOKE_OBJECTS_PER_CLASS,
    seed=SEED,
)
display(pd.crosstab(role_manifest["final_label"], role_manifest["role"], margins=True))

# Hash role membership so smoke and full databases cannot share incompatible caches.
database_config = {
    "sample": SAMPLE_ID,
    "split_strategy": SPLIT_STRATEGY,
    "execution_mode": EXECUTION_MODE,
    "object_ids": sorted(role_manifest["object_id"].astype(str)),
    "database_schema_version": supernova.DATABASE_SCHEMA_VERSION,
    "feature_order": list(supernova.ALL_FEATURES),
}
DATABASE_ID = workflow.configuration_hash(database_config)
DATABASE_PATH = RUNS_ROOT / SAMPLE_ID / SPLIT_STRATEGY / "supernnova" / "_data" / DATABASE_ID / "database.h5"
database_summary = supernova.prepare_supernnova_database(
    SAMPLE_DIR,
    truth,
    role_manifest,
    DATABASE_PATH,
    overwrite=FORCE_REBUILD_DATABASE,
)
display(pd.Series(database_summary).drop("feature_order"))
print(f"Sequence database: {DATABASE_PATH}")

## Train or reload both variants

The optimizer sees inverse-frequency weighted cross-entropy. Complete validation sequences select the best checkpoint by class-balanced log loss. Full runs halve the learning rate after five unimproved epochs and stop after twelve; the smoke run always has a two-epoch ceiling.

In [ ]:
# Train every requested seed and redshift variant, reusing complete caches by default.
runs = {}
for run_seed in RUN_SEEDS:
    for redshift_mode in REDSHIFT_MODES:
        training_config = supernova.SuperNNovaTrainingConfig(
            redshift_mode=redshift_mode,
            seed=run_seed,
            epochs=MAX_EPOCHS,
            device=DEVICE,
            threads=THREADS,
        )
        experiment_config = workflow.ExperimentConfig(
            training_sample=SAMPLE_ID,
            evaluation_sample=SAMPLE_ID,
            backend="supernnova",
            redshift_mode=redshift_mode,
            split_strategy=SPLIT_STRATEGY,
            seed=run_seed,
            model_config={
                **training_config.normalized(),
                "execution_mode": EXECUTION_MODE,
                "database_id": DATABASE_ID,
            },
        )
        run_dir = workflow.run_directory(RUNS_ROOT, experiment_config)
        print(f"\n=== SuperNNova {redshift_mode} (seed {run_seed}) ===")
        history = supernova.train_supernnova(
            DATABASE_PATH,
            run_dir / EXECUTION_MODE / "models",
            training_config,
            force=FORCE_RETRAIN,
            show_progress=True,
        )
        experiment_path = run_dir / EXECUTION_MODE / "experiment.json"
        if not experiment_path.exists() or FORCE_RETRAIN:
            experiment_metadata = workflow.build_experiment_metadata(
                experiment_config, split_manifest, status="smoke_complete" if EXECUTION_MODE == "smoke" else "trained_before_test"
            )
            experiment_metadata["database_summary"] = database_summary
            experiment_metadata["source_versions"] = supernova.supernnova_source_versions()
            workflow.write_json_once(experiment_metadata, experiment_path, overwrite=FORCE_RETRAIN)
        runs[(run_seed, redshift_mode)] = {
            "config": experiment_config,
            "training_config": training_config,
            "run_dir": run_dir,
            "history": history,
            "checkpoint": run_dir / EXECUTION_MODE / "models" / "best.pt",
        }
        print(redshift_mode, run_seed, f"{history['elapsed_seconds'] / 60:.2f} min", run_dir)

## Smoke validation and preliminary confusion matrices

Smoke validation uses fold 3 and the disposable smoke test uses fold 2. Neither is the official fold-0 test. With 64 objects per class, row-normalized confusion matrices make the preliminary class recall directly comparable across variants.

In [ ]:
# Score and compare both smoke variants without opening the official test fold.
smoke_predictions = {}
smoke_metrics = {}
if EXECUTION_MODE == "smoke":
    smoke_manifest = role_manifest.copy()
    smoke_manifest["split"] = smoke_manifest["role"].map({
        "train": "smoke_train", "validation": "smoke_validation", "test": "smoke_test"
    })
    for redshift_mode in REDSHIFT_MODES:
        run = runs[(SEED, redshift_mode)]
        output_dir = run["run_dir"] / EXECUTION_MODE
        for role, partition in (("validation", "smoke_validation"), ("test", "smoke_test")):
            prediction_path = output_dir / "predictions" / f"{partition}.parquet"
            metric_path = output_dir / "metrics" / f"{partition}.json"
            if prediction_path.exists() and metric_path.exists() and not FORCE_RETRAIN:
                predictions = pd.read_parquet(prediction_path)
                metrics = json.loads(metric_path.read_text())
            else:
                result = supernova.predict_supernnova(
                    run["checkpoint"], DATABASE_PATH, role=role, device=DEVICE
                )
                predictions = workflow.standardize_predictions(
                    result.classifications, smoke_manifest, run["config"], partition=partition
                )
                metrics = workflow.compute_classification_metrics(predictions)
                workflow.write_table_once(predictions, prediction_path, overwrite=FORCE_RETRAIN)
                workflow.write_json_once(metrics, metric_path, overwrite=FORCE_RETRAIN)
            smoke_predictions[(redshift_mode, partition)] = predictions
            smoke_metrics[(redshift_mode, partition)] = metrics

    from sklearn.metrics import confusion_matrix
    figure, axes = plt.subplots(2, 2, figsize=(15, 12))
    for row, redshift_mode in enumerate(REDSHIFT_MODES):
        for column, partition in enumerate(("smoke_validation", "smoke_test")):
            predictions = smoke_predictions[(redshift_mode, partition)]
            matrix = confusion_matrix(
                predictions["true_class"], predictions["predicted_class"],
                labels=workflow.FINAL_CLASSES, normalize="true"
            )
            sns.heatmap(
                matrix, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="Blues",
                xticklabels=workflow.FINAL_CLASSES, yticklabels=workflow.FINAL_CLASSES,
                ax=axes[row, column]
            )
            axes[row, column].set(
                title=f"{redshift_mode}: {partition.replace('_', ' ')}",
                xlabel="Predicted class", ylabel="True class"
            )
    figure.tight_layout()
    SMOKE_FIGURE_PATH = RUNS_ROOT / SAMPLE_ID / SPLIT_STRATEGY / "supernnova" / "smoke_comparison" / DATABASE_ID / "confusion_matrices.png"
    SMOKE_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(SMOKE_FIGURE_PATH, dpi=180, bbox_inches="tight")
    plt.close(figure)
    from IPython.display import Image
    display(Image(filename=str(SMOKE_FIGURE_PATH)))
    metric_rows = []
    for (redshift_mode, partition), metrics in smoke_metrics.items():
        metric_rows.append({
            "variant": redshift_mode, "partition": partition,
            **{key: metrics[key] for key in (
                "class_balanced_log_loss", "balanced_accuracy", "macro_f1",
                "top_1_accuracy", "top_2_accuracy", "multiclass_brier"
            )},
        })
    display(pd.DataFrame(metric_rows).set_index(["variant", "partition"]))
    print(f"Preliminary matrices: {SMOKE_FIGURE_PATH}")
    print("Official fold 0 remains untouched.")
else:
    print("Smoke plots are shown only when EXECUTION_MODE='smoke'.")

## Frozen full evaluation and partial light curves

This section remains inactive in the default smoke run. In a full mode, enabling `EVALUATE_TEST` writes immutable fold-0 probabilities, the shared metric suite and group bootstrap, observing-condition breakdowns, calibration diagnostics, and peak-relative partial classifications. An object with no measurement before a partial cutoff is counted as ineligible rather than padded with invented data.

In [ ]:
# Evaluate full-run checkpoints once, including partial sequences and diagnostic artifacts.
full_predictions = {}
partial_summaries = []
if EXECUTION_MODE != "smoke" and EVALUATE_TEST:
    test_observations = workflow.select_partition_rows(SAMPLE_DIR, split_manifest, "test")
    for key, run in runs.items():
        run_seed, redshift_mode = key
        output_dir = run["run_dir"] / EXECUTION_MODE
        prediction_path = output_dir / "predictions" / "test.parquet"
        metric_path = output_dir / "metrics" / "test.json"
        if prediction_path.exists() and metric_path.exists() and not ALLOW_TEST_OVERWRITE:
            predictions = pd.read_parquet(prediction_path)
            metrics = json.loads(metric_path.read_text())
        else:
            result = supernova.predict_supernnova(
                run["checkpoint"], DATABASE_PATH, role="test", device=DEVICE
            )
            predictions = workflow.standardize_predictions(
                result.classifications, split_manifest, run["config"], partition="test"
            )
            metrics = workflow.compute_classification_metrics(predictions)
            metrics["group_bootstrap_95"] = workflow.group_bootstrap_confidence_intervals(
                predictions, split_manifest, repeats=1000, seed=run_seed
            )
            breakdowns = workflow.metric_breakdowns(
                predictions, truth, test_observations
            )
            workflow.write_table_once(predictions, prediction_path, overwrite=ALLOW_TEST_OVERWRITE)
            workflow.write_json_once(metrics, metric_path, overwrite=ALLOW_TEST_OVERWRITE)
            workflow.write_table_once(
                breakdowns, output_dir / "metrics" / "test_breakdowns.parquet",
                overwrite=ALLOW_TEST_OVERWRITE
            )
        full_predictions[key] = predictions

        for cutoff in PARTIAL_CUTOFFS:
            result = supernova.predict_supernnova(
                run["checkpoint"], DATABASE_PATH, role="test", cutoff_days=cutoff, device=DEVICE
            )
            if result.eligible_objects:
                partial = workflow.standardize_predictions(
                    result.classifications, split_manifest, run["config"], partition="test"
                )
                partial_metrics = workflow.compute_classification_metrics(partial)
                partial["cutoff_days"] = cutoff
                workflow.write_table_once(
                    partial, output_dir / "predictions" / f"test_cutoff_{cutoff:+d}.parquet",
                    overwrite=ALLOW_TEST_OVERWRITE
                )
            else:
                partial_metrics = {}
            partial_summaries.append({
                "seed": run_seed, "variant": redshift_mode, "cutoff_days": cutoff,
                "eligible": result.eligible_objects, "ineligible": result.ineligible_objects,
                **{name: partial_metrics.get(name, np.nan) for name in (
                    "class_balanced_log_loss", "balanced_accuracy", "macro_f1", "top_2_accuracy"
                )},
            })
    partial_table = pd.DataFrame(partial_summaries)
    display(partial_table)
else:
    print("Full fold-0 evaluation is gated by a full execution mode and EVALUATE_TEST=True.")

In [ ]:
# Compare paired redshift variants and plot full-test calibration/confusion diagnostics.
if full_predictions:
    from sklearn.metrics import confusion_matrix
    for run_seed in RUN_SEEDS:
        first = full_predictions[(run_seed, "photometry_only")]
        second = full_predictions[(run_seed, "photometry_plus_truth_z")]
        paired = supernova.paired_group_bootstrap_differences(
            first, second, split_manifest, repeats=1000, seed=run_seed
        )
        comparison_dir = RUNS_ROOT / SAMPLE_ID / SPLIT_STRATEGY / "supernnova" / "comparisons"
        workflow.write_json_once(
            paired, comparison_dir / f"paired_seed{run_seed}.json",
            overwrite=ALLOW_TEST_OVERWRITE
        )
        display(pd.DataFrame(paired["intervals"]).T)

    for key, predictions in full_predictions.items():
        run = runs[key]
        metrics = workflow.compute_classification_metrics(predictions)
        probabilities = predictions[[f"prob_{label}" for label in workflow.FINAL_CLASSES]].to_numpy()
        entropy = -(probabilities * np.log(np.clip(probabilities, 1e-15, 1))).sum(axis=1)
        raw = confusion_matrix(
            predictions["true_class"], predictions["predicted_class"], labels=workflow.FINAL_CLASSES
        )
        normalized = confusion_matrix(
            predictions["true_class"], predictions["predicted_class"],
            labels=workflow.FINAL_CLASSES, normalize="true"
        )
        figure, axes = plt.subplots(2, 2, figsize=(14, 11))
        sns.heatmap(raw, annot=True, fmt="d", xticklabels=workflow.FINAL_CLASSES, yticklabels=workflow.FINAL_CLASSES, ax=axes[0, 0])
        sns.heatmap(normalized, annot=True, fmt=".2f", vmin=0, vmax=1, xticklabels=workflow.FINAL_CLASSES, yticklabels=workflow.FINAL_CLASSES, ax=axes[0, 1])
        calibration = pd.DataFrame(metrics["calibration"])
        axes[1, 0].plot([0, 1], [0, 1], "k--", label="Ideal")
        axes[1, 0].plot(calibration["confidence"], calibration["accuracy"], "o-", label="Model")
        axes[1, 0].set(xlabel="Mean confidence", ylabel="Observed accuracy", xlim=(0, 1), ylim=(0, 1))
        axes[1, 0].legend()
        sns.histplot(entropy, bins=30, ax=axes[1, 1])
        axes[1, 1].set(xlabel="Predictive entropy")
        figure.suptitle(f"SuperNNova {key[1]} seed {key[0]}")
        figure.tight_layout()
        figure_dir = run["run_dir"] / EXECUTION_MODE / "figures"
        figure_dir.mkdir(parents=True, exist_ok=True)
        figure.savefig(figure_dir / "test_confusion_calibration_entropy.png", dpi=180, bbox_inches="tight")
else:
    print("Paired and full-test diagnostic plots remain disabled until frozen test predictions exist.")

## Interpretation

The smoke result verifies preprocessing, optimization, checkpoint reload, and standardized probabilities; two epochs are not expected to converge. Compare full scientific results only when the evaluation sample and split strategy match, and treat `photometry_plus_truth_z` as an optimistic exact-redshift baseline.